# 02 — Feature Engineering: Biến Đổi Wavelet & Đặc Trưng Toán Học

Notebook này thực hiện:
1. Tải dữ liệu đã làm sạch từ `data/processed/steel_clean.csv`
2. Trích xuất đặc trưng tần số-thời gian bằng Discrete Wavelet Transform (DWT)
3. Tính đặc trưng Wavelet theo cửa sổ trượt (rolling window)
4. Phân tích và trực quan hóa hệ số Wavelet
5. Lưu tập đặc trưng đầy đủ vào `data/processed/`

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pywt
from pathlib import Path

from src.data_loader import load_data
from src.wavelet_features import (
    extract_wavelet_features,
    apply_wavelet_features_to_df,
    rolling_wavelet_features,
)
from src.utils import save_data

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 1. Tải dữ liệu đã làm sạch

In [ ]:
from src.utils import load_data

CLEAN_PATH = Path('data/processed/steel_clean.csv')
df = load_data(CLEAN_PATH)
df['date'] = pd.to_datetime(df['date'])
print(f'Shape: {df.shape}')
df.head()

## 2. Phân tích Wavelet đơn giản — Phân rã DWT

In [ ]:
TARGET_COL = 'Usage_kWh'
series = df[TARGET_COL].values

# Phân rã Wavelet 3 mức, dùng wavelet Daubechies bậc 4
wavelet = 'db4'
level = 3
coeffs = pywt.wavedec(series, wavelet=wavelet, level=level)

labels = ['Approximation (cA3)'] + [f'Detail Level {level - i} (cD{level - i})' for i in range(1, level + 1)]

fig, axes = plt.subplots(level + 1, 1, figsize=(14, 3 * (level + 1)))
for ax, coeff, label in zip(axes, coeffs, labels):
    ax.plot(coeff, linewidth=0.7)
    ax.set_title(label)
plt.suptitle(f'Wavelet Decomposition of {TARGET_COL} ({wavelet}, level={level})', y=1.01)
plt.tight_layout()
plt.show()

## 3. Đặc trưng Wavelet tổng hợp cho toàn bộ tập dữ liệu

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
global_features = apply_wavelet_features_to_df(df, target_cols=numeric_cols)
print('Global Wavelet features:')
print(global_features)

## 4. Đặc trưng Wavelet theo cửa sổ trượt

In [ ]:
WINDOW = 64
rolling_feats = rolling_wavelet_features(
    df, target_col=TARGET_COL, window=WINDOW, wavelet=wavelet, level=level
)
print(f'Rolling wavelet features shape: {rolling_feats.shape}')
rolling_feats.head(10)

## 5. Kết hợp và lưu tập đặc trưng

In [ ]:
df_featured = pd.concat([df, rolling_feats], axis=1)
print(f'Total features: {len(df_featured.columns)}')

OUTPUT_PATH = Path('data/processed/steel_wavelet_features.csv')
save_data(df_featured, OUTPUT_PATH)
print(f'Saved to {OUTPUT_PATH}')